# 01 · Validación, limpieza, EDA y entrega de base para modelado

Este notebook prepara la base de datos inicial del proyecto **BiciMAD Predictor**.

El objetivo es validar las fuentes de datos, limpiar la información necesaria, integrar variables temporales y meteorológicas, y realizar un análisis exploratorio que ayude a entender los principales riesgos operativos de las estaciones de BiciMAD.

Este notebook también deja documentado qué archivo debe utilizarse como punto de partida para la fase de preprocesamiento y modelado.

### Qué hace este notebook

- Identifica las fuentes de datos utilizadas.
- Explica para qué sirve cada dataset.
- Limpia y valida datos históricos de BiciMAD.
- Integra calendario laboral de Madrid.
- Integra meteorología histórica de AEMET.
- Crea una base enriquecida intermedia.
- Define reglas de limpieza para el análisis exploratorio.
- Analiza riesgos de estaciones casi vacías y casi llenas.
- Deja claro qué base debe usar el siguiente notebook.

### Qué no hace este notebook

- No entrena modelos.
- No define el modelo final.
- No calcula métricas predictivas.
- No realiza división train/test.
- No aplica codificación ni escalado final de variables.
- No genera el dataset final de Machine Learning en `data/processed`.

La base generada aquí será el punto de partida para que la fase de preprocesamiento y modelado tome sus propias decisiones.


## Índice maestro del notebook

0. Portada y objetivo del notebook  
1. Resumen rápido para el equipo de modelado  
2. Explicación sencilla del problema y del dataset principal  
3. Mapa general del flujo de datos  
4. Inventario de fuentes originales  
5. Qué papel cumple cada dataset  
6. Configuración del proyecto  
7. Seguridad, credenciales y archivos protegidos  
8. Descarga o verificación de datos originales  
9. Validación inicial de datos raw  
10. Limpieza de datasets auxiliares  
    10.1 GBFS estaciones actuales  
    10.2 GBFS estado actual  
    10.3 Histórico diario de viajes  
    10.4 Calendario laboral  
    10.5 Meteorología AEMET  
11. Limpieza del histórico principal BiciMAD 2022  
12. Creación de la base enriquecida intermedia  
13. Validación de la base enriquecida  
14. Contrato de datos para modelado  
15. Reglas de limpieza para EDA  
16. EDA global  
17. EDA por estación  
18. EDA temporal  
19. EDA meteorológica  
20. EDA estación + hora  
21. Hallazgos principales  
22. Limitaciones  
23. Entrega para el siguiente notebook  
24. Archivos generados


## 1. Resumen rápido para el equipo de modelado

La base principal que este notebook deja preparada para las siguientes fases es:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`

Esta base contiene registros históricos de estaciones de BiciMAD durante 2022, enriquecidos con calendario laboral de Madrid y meteorología diaria observada de AEMET.

Cada fila representa el estado de una estación concreta de BiciMAD en un momento concreto del tiempo.

Dicho de forma sencilla:

> Cada fila responde a la pregunta:  
> **¿Cómo estaba una estación concreta de BiciMAD en una fecha y hora concretas?**

Esta base todavía **no es el dataset final de Machine Learning**. No contiene una variable objetivo definitiva, no tiene división train/test y no tiene codificación ni escalado final de variables.

El siguiente notebook deberá partir de esta base para definir el problema predictivo concreto, crear el target, seleccionar variables, dividir los datos temporalmente y entrenar modelos.


### 1.1 Base recomendada para la fase de preprocesamiento y modelado

| Elemento | Valor |
|---|---|
| Archivo base recomendado | `data/interim/bicimad/station_status_history_2022_enriched_base.csv` |
| Granularidad | Estación + momento temporal |
| Identificador principal | `station_id_historical` |
| Fecha/hora principal | `snapshot_datetime` |
| Variables temporales | `date`, `snapshot_hour`, `snapshot_day_of_week`, `snapshot_month` |
| Variables de calendario | `day_type`, `is_working_day`, `is_weekend`, `is_holiday` |
| Variables meteorológicas | temperatura, precipitación, humedad |
| Variables operativas | `num_bikes_available`, `num_docks_available`, `occupation_ratio` |
| Ubicación del archivo | `data/interim`, no `data/processed` |
| Responsabilidad del siguiente notebook | crear target, preprocesar variables, dividir datos y entrenar modelos |

Este notebook no decide el target final ni entrena modelos. Su responsabilidad es dejar una base histórica limpia, validada, enriquecida y bien documentada para que el siguiente paso pueda trabajar con seguridad.


## 2. Explicación sencilla del problema y del dataset principal

BiciMAD es un sistema de bicicleta pública. En este tipo de sistema pueden aparecer dos problemas operativos principales:

1. Una estación puede quedarse con muy pocas bicicletas disponibles.
2. Una estación puede quedarse con muy pocos anclajes libres para devolver bicicletas.

Estos problemas no ocurren igual en todas las estaciones ni en todas las horas. Algunas estaciones pueden quedarse casi vacías en determinadas franjas horarias, mientras que otras pueden saturarse y dificultar la devolución de bicicletas.

Por eso, el proyecto necesita una base histórica que permita estudiar patrones por:

- estación,
- fecha,
- hora,
- tipo de día,
- condiciones meteorológicas,
- disponibilidad de bicicletas,
- disponibilidad de anclajes.

La base principal del proyecto se construye a partir del histórico de estado de estaciones de BiciMAD de 2022. A ese histórico se le añaden variables de calendario y meteorología para que el análisis y el futuro modelo tengan más contexto.


## 3. Mapa general del flujo de datos

El camino de los datos en este notebook sigue esta lógica:

~~~text
Datos originales
│
├── Histórico de estaciones BiciMAD 2022
│   └── Limpieza
│       └── station_status_history_2022_clean.csv
│
├── Calendario laboral de Madrid
│   └── Limpieza
│       └── madrid_working_calendar_clean.csv
│
├── Clima diario AEMET 2022
│   └── Limpieza y agregación
│       └── aemet_daily_weather_madrid_2022_join_ready.csv
│
└── Unión por fecha
    └── station_status_history_2022_enriched_base.csv
        └── Entrada recomendada para preprocesamiento y modelado
~~~

No todos los datasets descargados se usan para entrenar un modelo. Algunos sirven para enriquecer el histórico, otros para contexto de negocio y otros para una posible demo futura.

El dataset que conecta directamente con la fase de modelado es la base enriquecida histórica de estaciones:

`data/interim/bicimad/station_status_history_2022_enriched_base.csv`


## 4. Inventario de fuentes originales

Antes de limpiar o transformar datos, es importante saber de dónde viene cada fuente y para qué sirve.

Esta sección crea un inventario de datasets del proyecto. La tabla distingue entre:

- fuentes principales para análisis y modelado,
- fuentes auxiliares para enriquecer datos,
- fuentes útiles para contexto, visualización o demo futura.

El objetivo es evitar una confusión habitual: pensar que todos los datos descargados se usan directamente para entrenar el modelo. En este proyecto no es así.


In [30]:
from dataclasses import dataclass
from pathlib import Path
import os
import sys
import platform
import pandas as pd
from IPython.display import display


@dataclass
class DataSource:
    """Representa una fuente de datos utilizada en el proyecto."""

    name: str
    source_type: str
    origin: str
    granularity: str
    project_use: str
    raw_file: str
    interim_file: str
    modeling_role: str


class DataSourceRegistry:
    """Inventario de fuentes de datos del proyecto."""

    def __init__(self, sources: list[DataSource]):
        self.sources = sources

    def to_dataframe(self) -> pd.DataFrame:
        """Devuelve el inventario en formato tabla."""
        return pd.DataFrame(
            [
                {
                    "Dataset": source.name,
                    "Tipo": source.source_type,
                    "Origen": source.origin,
                    "Granularidad": source.granularity,
                    "Uso en el proyecto": source.project_use,
                    "Archivo raw": source.raw_file,
                    "Archivo intermedio": source.interim_file,
                    "Papel para modelado": source.modeling_role,
                }
                for source in self.sources
            ]
        )


data_sources = [
    DataSource(
        name="Histórico de estaciones BiciMAD 2022",
        source_type="Fuente principal",
        origin="EMT Madrid / BiciMAD",
        granularity="Estación + momento temporal",
        project_use="Base principal para analizar disponibilidad y saturación por estación.",
        raw_file="data/raw/bicimad/bicimad_station_status_history_2022_raw.zip",
        interim_file="data/interim/bicimad/station_status_history_2022_clean.csv",
        modeling_role="Fuente base para construir el dataset de modelado.",
    ),
    DataSource(
        name="Calendario laboral de Madrid",
        source_type="Fuente auxiliar",
        origin="Portal de datos abiertos del Ayuntamiento de Madrid",
        granularity="Día",
        project_use="Añadir contexto de día laborable, sábado, domingo o festivo.",
        raw_file="data/raw/calendar/madrid_working_calendar_raw.csv",
        interim_file="data/interim/calendar/madrid_working_calendar_clean.csv",
        modeling_role="Enriquece la base principal con variables temporales.",
    ),
    DataSource(
        name="Climatología diaria AEMET 2022",
        source_type="Fuente auxiliar",
        origin="AEMET OpenData",
        granularity="Día + estación meteorológica",
        project_use="Añadir temperatura, precipitación y humedad observada al histórico 2022.",
        raw_file="data/raw/weather/aemet_daily_climate_selected_stations_2022_raw.json",
        interim_file="data/interim/weather/aemet_daily_weather_madrid_2022_join_ready.csv",
        modeling_role="Enriquece la base principal con variables meteorológicas históricas.",
    ),
    DataSource(
        name="GBFS estaciones actuales BiciMAD",
        source_type="Fuente de referencia actual",
        origin="GBFS BiciMAD",
        granularity="Estación actual",
        project_use="Referencia actual de estaciones, coordenadas, capacidad y atributos.",
        raw_file="data/raw/bicimad/gbfs_station_information_raw.json",
        interim_file="data/interim/bicimad/stations_clean.csv",
        modeling_role="No es la base de entrenamiento histórica; útil para demo, mapa o referencia actual.",
    ),
    DataSource(
        name="GBFS estado actual BiciMAD",
        source_type="Fuente de referencia actual",
        origin="GBFS BiciMAD",
        granularity="Foto actual de estación",
        project_use="Consultar disponibilidad actual de bicicletas y anclajes.",
        raw_file="data/raw/bicimad/gbfs_station_status_snapshot_raw.json",
        interim_file="data/interim/bicimad/station_status_snapshot_clean.csv",
        modeling_role="No se usa para entrenar el histórico 2022; útil para demo o comparación futura.",
    ),
    DataSource(
        name="Histórico diario de viajes BiciMAD",
        source_type="Fuente de contexto",
        origin="EMT Madrid / BiciMAD",
        granularity="Día",
        project_use="Entender la evolución general del uso de BiciMAD.",
        raw_file="data/raw/bicimad/bicimad_daily_trips_raw.csv",
        interim_file="data/interim/bicimad/bicimad_daily_trips_clean.csv",
        modeling_role="Contexto de negocio; no sustituye al histórico por estación.",
    ),
    DataSource(
        name="Inventario de estaciones AEMET",
        source_type="Fuente auxiliar",
        origin="AEMET OpenData",
        granularity="Estación meteorológica",
        project_use="Seleccionar estaciones meteorológicas cercanas y fiables para Madrid.",
        raw_file="data/raw/weather/aemet_stations_inventory_raw.json",
        interim_file="data/interim/weather/aemet_stations_inventory_clean.csv",
        modeling_role="Apoya la preparación de variables meteorológicas históricas.",
    ),
    DataSource(
        name="Predicción horaria AEMET Madrid",
        source_type="Fuente para demo futura",
        origin="AEMET OpenData",
        granularity="Hora futura",
        project_use="Demostrar cómo una app futura podría consumir meteorología prevista.",
        raw_file="data/raw/weather/aemet_forecast_hourly_madrid_raw.json",
        interim_file="data/interim/weather/aemet_forecast_hourly_madrid_clean.csv",
        modeling_role="No se usa para entrenar el histórico 2022; sirve para demo o integración futura.",
    ),
]

registry = DataSourceRegistry(data_sources)
data_sources_df = registry.to_dataframe()

display(data_sources_df)


,Dataset,Tipo,Origen,Granularidad,Uso en el proyecto,Archivo raw,Archivo intermedio,Papel para modelado
0,Histórico de estaciones BiciMAD 2022,Fuente principal,EMT Madrid / BiciMAD,Estación + momento temporal,Base principal para analizar disponibilidad y ...,data/raw/bicimad/bicimad_station_status_histor...,data/interim/bicimad/station_status_history_20...,Fuente base para construir el dataset de model...
1,Calendario laboral de Madrid,Fuente auxiliar,Portal de datos abiertos del Ayuntamiento de M...,Día,"Añadir contexto de día laborable, sábado, domi...",data/raw/calendar/madrid_working_calendar_raw.csv,data/interim/calendar/madrid_working_calendar_...,Enriquece la base principal con variables temp...
2,Climatología diaria AEMET 2022,Fuente auxiliar,AEMET OpenData,Día + estación meteorológica,"Añadir temperatura, precipitación y humedad ob...",data/raw/weather/aemet_daily_climate_selected_...,data/interim/weather/aemet_daily_weather_madri...,Enriquece la base principal con variables mete...
3,GBFS estaciones actuales BiciMAD,Fuente de referencia actual,GBFS BiciMAD,Estación actual,"Referencia actual de estaciones, coordenadas, ...",data/raw/bicimad/gbfs_station_information_raw....,data/interim/bicimad/stations_clean.csv,No es la base de entrenamiento histórica; útil...
4,GBFS estado actual BiciMAD,Fuente de referencia actual,GBFS BiciMAD,Foto actual de estación,Consultar disponibilidad actual de bicicletas ...,data/raw/bicimad/gbfs_station_status_snapshot_...,data/interim/bicimad/station_status_snapshot_c...,No se usa para entrenar el histórico 2022; úti...
5,Histórico diario de viajes BiciMAD,Fuente de contexto,EMT Madrid / BiciMAD,Día,Entender la evolución general del uso de BiciMAD.,data/raw/bicimad/bicimad_daily_trips_raw.csv,data/interim/bicimad/bicimad_daily_trips_clean...,Contexto de negocio; no sustituye al histórico...
6,Inventario de estaciones AEMET,Fuente auxiliar,AEMET OpenData,Estación meteorológica,Seleccionar estaciones meteorológicas cercanas...,data/raw/weather/aemet_stations_inventory_raw....,data/interim/weather/aemet_stations_inventory_...,Apoya la preparación de variables meteorológic...
7,Predicción horaria AEMET Madrid,Fuente para demo futura,AEMET OpenData,Hora futura,Demostrar cómo una app futura podría consumir ...,data/raw/weather/aemet_forecast_hourly_madrid_...,data/interim/weather/aemet_forecast_hourly_mad...,No se usa para entrenar el histórico 2022; sir...


## 5. Qué papel cumple cada dataset

El proyecto utiliza varios datasets, pero no todos tienen el mismo papel.

La fuente central es el histórico de estado de estaciones de BiciMAD 2022. Esta fuente indica cuántas bicicletas y anclajes había disponibles en cada estación a lo largo del tiempo.

El calendario laboral y la meteorología diaria no sustituyen a BiciMAD, sino que lo enriquecen. Sirven para añadir contexto: no es lo mismo analizar una estación en un día laborable que en un festivo, ni en un día frío que en un día templado.

Las fuentes GBFS actuales representan una fotografía reciente de la red. Son útiles para una demo, un mapa o una aplicación futura, pero no deben mezclarse automáticamente con el histórico de 2022 sin un trabajo adicional de compatibilidad.

La predicción horaria de AEMET es útil para una posible app futura, porque permite consultar el tiempo previsto. Sin embargo, no se usa para entrenar el histórico de 2022, ya que el entrenamiento necesita variables históricas observadas.

En resumen:

- **Dataset principal para el siguiente paso:** histórico enriquecido de estaciones BiciMAD 2022.
- **Datasets auxiliares:** calendario laboral y clima diario observado.
- **Datasets de contexto o demo:** GBFS actual, viajes diarios y predicción horaria.


## 6. Configuración del proyecto

Antes de descargar o transformar datos, necesitamos comprobar que el notebook se está ejecutando dentro del repositorio correcto y que las carpetas principales existen.

Para evitar rutas escritas manualmente muchas veces, se define una clase de configuración central. Esta clase guarda las rutas importantes del proyecto:

- carpeta raíz del repositorio,
- carpeta `data/raw`,
- carpeta `data/interim`,
- carpeta `data/processed`,
- carpeta `notebooks`,
- carpeta `src`,
- carpeta `models`.

Esta estructura ayuda a que el notebook sea reproducible para cualquier persona del equipo que clone el repositorio.


In [31]:
@dataclass
class ProjectConfig:
    """Configuración central de rutas del proyecto."""

    project_root: Path

    @property
    def data_dir(self) -> Path:
        return self.project_root / "data"

    @property
    def raw_dir(self) -> Path:
        return self.data_dir / "raw"

    @property
    def interim_dir(self) -> Path:
        return self.data_dir / "interim"

    @property
    def processed_dir(self) -> Path:
        return self.data_dir / "processed"

    @property
    def notebooks_dir(self) -> Path:
        return self.project_root / "notebooks"

    @property
    def src_dir(self) -> Path:
        return self.project_root / "src"

    @property
    def models_dir(self) -> Path:
        return self.project_root / "models"

    @property
    def raw_bicimad_dir(self) -> Path:
        return self.raw_dir / "bicimad"

    @property
    def raw_calendar_dir(self) -> Path:
        return self.raw_dir / "calendar"

    @property
    def raw_weather_dir(self) -> Path:
        return self.raw_dir / "weather"

    @property
    def interim_bicimad_dir(self) -> Path:
        return self.interim_dir / "bicimad"

    @property
    def interim_calendar_dir(self) -> Path:
        return self.interim_dir / "calendar"

    @property
    def interim_weather_dir(self) -> Path:
        return self.interim_dir / "weather"

    @property
    def expected_directories(self) -> list[Path]:
        return [
            self.data_dir,
            self.raw_dir,
            self.raw_bicimad_dir,
            self.raw_calendar_dir,
            self.raw_weather_dir,
            self.interim_dir,
            self.interim_bicimad_dir,
            self.interim_calendar_dir,
            self.interim_weather_dir,
            self.processed_dir,
            self.notebooks_dir,
            self.src_dir,
            self.models_dir,
        ]


### 6.1 Detección de la raíz del repositorio

Esta celda busca automáticamente la carpeta raíz del proyecto.

La raíz del proyecto se identifica porque contiene la carpeta `.git`. Esto permite ejecutar el notebook desde distintas ubicaciones sin romper las rutas relativas.

Si esta celda falla, normalmente significa que el notebook no se está ejecutando dentro del repositorio clonado.


In [32]:
class ProjectRootFinder:
    """Localiza la raíz del repositorio a partir de la carpeta actual."""

    @staticmethod
    def find(start_path: Path | None = None) -> Path:
        current_path = start_path or Path.cwd()

        for path in [current_path, *current_path.parents]:
            if (path / ".git").exists():
                return path

        raise FileNotFoundError(
            "No se ha podido localizar la raíz del repositorio. "
            "Asegúrate de ejecutar este notebook dentro del proyecto clonado."
        )


project_root = ProjectRootFinder.find()
config = ProjectConfig(project_root=project_root)

print("Raíz del proyecto detectada correctamente:")
print(config.project_root)


Raíz del proyecto detectada correctamente:
/mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo3/1_Proyecto/bike-sharing-workspace/bike-sharing-prediction


### 6.2 Información básica del entorno

Esta celda muestra información básica del entorno de ejecución.

No modifica ningún archivo. Solo ayuda a detectar diferencias entre ordenadores, entornos de Python o rutas de trabajo.


In [33]:
print("Versión de Python:")
print(sys.version)

print("\nSistema operativo:")
print(platform.platform())

print("\nCarpeta actual de ejecución:")
print(Path.cwd())

print("\nRaíz del proyecto:")
print(config.project_root)


Versión de Python:
3.11.15 (main, Jun 11 2026, 15:20:16) [GCC 14.3.0]

Sistema operativo:
Linux-6.8.0-124-generic-x86_64-with-glibc2.35

Carpeta actual de ejecución:
/mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo3/1_Proyecto/bike-sharing-workspace/bike-sharing-prediction/notebooks

Raíz del proyecto:
/mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo3/1_Proyecto/bike-sharing-workspace/bike-sharing-prediction


### 6.3 Comprobación y creación controlada de carpetas del proyecto

Esta celda revisa si existen las carpetas necesarias para trabajar con datos y crea las que falten.

La estructura esperada separa los datos en tres niveles:

- `data/raw`: datos originales descargados sin modificar.
- `data/interim`: datos limpios, transformados o enriquecidos.
- `data/processed`: datasets finales preparados para modelado.

En este notebook se usará principalmente `data/raw` y `data/interim`.

La carpeta `data/processed` queda reservada para fases posteriores de preprocesamiento y modelado.


In [34]:
class ProjectStructureManager:
    """Comprueba y crea la estructura mínima de carpetas del proyecto."""

    def __init__(self, config: ProjectConfig):
        self.config = config

    def check_directories(self) -> pd.DataFrame:
        records = []

        for directory in self.config.expected_directories:
            records.append(
                {
                    "Carpeta": str(directory.relative_to(self.config.project_root)),
                    "Existe": directory.exists(),
                    "Es carpeta": directory.is_dir() if directory.exists() else False,
                }
            )

        return pd.DataFrame(records)

    def create_missing_directories(self) -> None:
        for directory in self.config.expected_directories:
            directory.mkdir(parents=True, exist_ok=True)


structure_manager = ProjectStructureManager(config)

structure_manager.create_missing_directories()
directory_check = structure_manager.check_directories()

print("Comprobación de carpetas del proyecto:")
display(directory_check)


Comprobación de carpetas del proyecto:


,Carpeta,Existe,Es carpeta
0,data,True,True
1,data/raw,True,True
2,data/raw/bicimad,True,True
3,data/raw/calendar,True,True
4,data/raw/weather,True,True
5,data/interim,True,True
6,data/interim/bicimad,True,True
7,data/interim/calendar,True,True
8,data/interim/weather,True,True
9,data/processed,True,True


## 7. Seguridad, credenciales y archivos protegidos

El proyecto puede usar la API de AEMET para descargar datos meteorológicos.

La clave real de AEMET debe guardarse en un archivo local llamado `.env`.

Ese archivo no debe subirse a GitHub porque contiene información privada. Por eso el repositorio incluye `.env.example`, que sirve como plantilla para que cada persona cree su propio `.env`.

Este notebook nunca imprime la clave real. Solo comprueba si existe y si parece estar configurada.


### 7.1 Cómo obtener y configurar la API Key de AEMET

Para descargar datos meteorológicos desde AEMET OpenData es necesario tener una API Key.

La API Key es una clave personal que permite hacer peticiones a la API de AEMET. No debe escribirse directamente en el código ni subirse a GitHub.

AEMET indica que la API Key se solicita desde la opción **“Obtención de API Key”** de AEMET OpenData introduciendo una dirección de correo electrónico.

Página oficial de AEMET OpenData:  
https://opendata.aemet.es/

Página oficial para obtener la API Key:  
https://opendata.aemet.es/centrodedescargas/obtencionAPIKey

Una vez recibida la clave, cada persona del equipo debe crear localmente un archivo `.env` en la raíz del repositorio.

Este archivo debe contener la clave, pero no debe subirse nunca a GitHub.

El repositorio incluye `.env.example` como plantilla, pero la clave real debe estar solo en `.env`.


### 7.2 Crear el archivo `.env` de forma segura

La siguiente configuración debe hacerse en la **terminal**, desde la raíz del repositorio.

No se recomienda pedir la API Key desde una celda de notebook porque podría quedar guardada accidentalmente en el historial, en la salida del notebook o en capturas de pantalla.

Ejecuta este bloque en la terminal:

~~~bash
read -r -s -p "Introduce tu AEMET_API_KEY: " AEMET_API_KEY
echo
umask 077
printf "AEMET_API_KEY=%s\n" "$AEMET_API_KEY" > .env
unset AEMET_API_KEY
chmod 600 .env
echo ".env creado con permisos seguros."
ls -l .env
~~~

Al pegar la clave, no se verá en pantalla. Esto es normal y es más seguro.

El archivo `.env` debe quedar con permisos similares a:

~~~text
-rw------- 1 usuario usuario ... .env
~~~

Esto significa que solo el usuario local puede leer y escribir ese archivo.


### 7.3 Verificar que `.env` está configurado sin mostrar la clave

Después de crear `.env`, se puede verificar que existe y que contiene una clave sin imprimir el valor real.

Ejecuta este bloque en la terminal:

~~~bash
python - <<'PY'
from pathlib import Path

env_path = Path(".env")

api_key = None

if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)

        if key.strip() == "AEMET_API_KEY":
            api_key = value.strip()

print(".env existe:", env_path.exists())

if env_path.exists():
    print("Permisos:", oct(env_path.stat().st_mode & 0o777))

print("AEMET_API_KEY configurada:", bool(api_key))
print("Longitud de AEMET_API_KEY:", len(api_key) if api_key else 0)
PY
~~~

La salida esperada debería indicar:

~~~text
.env existe: True
Permisos: 0o600
AEMET_API_KEY configurada: True
Longitud de AEMET_API_KEY: valor_mayor_que_0
~~~

La clave real no debe aparecer en pantalla.


### 7.4 Confirmar que `.env` no aparece en Git

Después de crear `.env`, conviene comprobar que Git no intenta subirlo al repositorio.

Ejecuta en terminal:

~~~bash
git status -sb
~~~

Si `.gitignore` está bien configurado, `.env` no debe aparecer como archivo pendiente.

Esto es importante porque `.env` contiene una clave privada local.


### 7.5 Comprobación segura de credenciales y archivos protegidos

Esta celda comprueba si `.env` existe, si `.env.example` existe y si la variable `AEMET_API_KEY` parece estar configurada.

También revisa que `.gitignore` contenga patrones importantes para proteger credenciales, datos generados y archivos temporales.

La clave real no se muestra en ningún momento.


In [35]:
class EnvironmentManager:
    """Gestiona comprobaciones básicas de archivos de entorno sin mostrar secretos."""

    def __init__(self, config: ProjectConfig):
        self.config = config
        self.env_path = self.config.project_root / ".env"
        self.env_example_path = self.config.project_root / ".env.example"

    def env_file_exists(self) -> bool:
        return self.env_path.exists()

    def env_example_exists(self) -> bool:
        return self.env_example_path.exists()

    def read_env_variable(self, variable_name: str) -> str | None:
        if not self.env_file_exists():
            return None

        for line in self.env_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            if "=" not in line:
                continue

            key, value = line.split("=", 1)

            if key.strip() == variable_name:
                return value.strip()

        return None

    def check_aemet_api_key(self) -> dict:
        api_key = self.read_env_variable("AEMET_API_KEY")

        return {
            ".env existe": self.env_file_exists(),
            ".env.example existe": self.env_example_exists(),
            "AEMET_API_KEY configurada": bool(api_key),
            "Longitud de AEMET_API_KEY": len(api_key) if api_key else 0,
        }


class GitIgnoreChecker:
    """Comprueba que .gitignore protege los elementos críticos del proyecto."""

    def __init__(self, config: ProjectConfig):
        self.gitignore_path = config.project_root / ".gitignore"

    def check_patterns(self, patterns: list[str]) -> pd.DataFrame:
        content = self.gitignore_path.read_text(encoding="utf-8") if self.gitignore_path.exists() else ""

        return pd.DataFrame(
            [
                {
                    "Patrón": pattern,
                    "Encontrado en .gitignore": pattern in content,
                }
                for pattern in patterns
            ]
        )


environment_manager = EnvironmentManager(config)
environment_status = environment_manager.check_aemet_api_key()

print("Estado de configuración de AEMET:")
display(pd.DataFrame([environment_status]))

if environment_status["AEMET_API_KEY configurada"]:
    print("La clave de AEMET está configurada localmente. No se muestra por seguridad.")
else:
    print("La clave de AEMET no está configurada todavía en .env.")

protected_patterns = [
    ".env",
    "data/raw/**",
    "data/interim/**",
    "data/processed/**",
    "models/**",
    ".ipynb_checkpoints/",
]

gitignore_checker = GitIgnoreChecker(config)
gitignore_status = gitignore_checker.check_patterns(protected_patterns)

print("\nComprobación de patrones críticos en .gitignore:")
display(gitignore_status)


Estado de configuración de AEMET:


,.env existe,.env.example existe,AEMET_API_KEY configurada,Longitud de AEMET_API_KEY
0,True,True,True,293


La clave de AEMET está configurada localmente. No se muestra por seguridad.

Comprobación de patrones críticos en .gitignore:


,Patrón,Encontrado en .gitignore
0,.env,True
1,data/raw/**,True
2,data/interim/**,True
3,data/processed/**,True
4,models/**,True
5,.ipynb_checkpoints/,True


### 7.6 Resumen de seguridad y configuración

En este punto el notebook ya ha comprobado:

- la raíz del repositorio,
- la estructura de carpetas,
- la existencia de `.env.example`,
- si existe una clave local de AEMET,
- si `.gitignore` protege datos y credenciales.

A partir de la siguiente sección se podrá empezar con la descarga o verificación de datos originales.


In [36]:
print("Configuración inicial completada.")
print(f"Raíz del proyecto: {config.project_root}")
print(f"Datos originales: {config.raw_dir}")
print(f"Datos intermedios: {config.interim_dir}")
print(f"Datos finales reservados para modelado: {config.processed_dir}")
print(f"Notebooks: {config.notebooks_dir}")
print(f"Modelos: {config.models_dir}")


Configuración inicial completada.
Raíz del proyecto: /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo3/1_Proyecto/bike-sharing-workspace/bike-sharing-prediction
Datos originales: /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo3/1_Proyecto/bike-sharing-workspace/bike-sharing-prediction/data/raw
Datos intermedios: /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo3/1_Proyecto/bike-sharing-workspace/bike-sharing-prediction/data/interim
Datos finales reservados para modelado: /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo3/1_Proyecto/bike-sharing-workspace/bike-sharing-prediction/data/processed
Notebooks: /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo3/1_Proyecto/bike-sharing-workspace/bike-sharing-prediction/notebooks
Modelos: /mnt/hdd/David/2026/Somos_F5/Bootcam_Desarrollo_IA/Proyectos/Modulo3/1_Proyecto/bike-sharing-workspace/bike-sharing-prediction/models


## 8. Descarga o verificación de datos originales

En esta sección se prepara el control de los archivos originales del proyecto.

Los datos originales se guardan en `data/raw` y no se modifican directamente. La idea es conservar una copia lo más fiel posible a la fuente original.

Este apartado no limpia datos todavía. Solo responde a estas preguntas:

- Qué archivos originales necesita el proyecto.
- De dónde viene cada archivo.
- Dónde se debe guardar localmente.
- Si el archivo ya existe o falta.
- Qué archivos son pesados.
- Qué archivos requieren API Key.

Por defecto, el notebook no descarga datos automáticamente. Primero verifica la disponibilidad local y después permite activar descargas de forma explícita.


### 8.1 Parámetros de ejecución

Algunas fuentes son pequeñas y pueden descargarse rápido. Otras, como el histórico de BiciMAD 2022, pueden ocupar cientos de MB.

Para evitar descargas accidentales, se usan tres parámetros:

- `RUN_DOWNLOADS`: activa o desactiva descargas generales.
- `RUN_HEAVY_DOWNLOADS`: activa o desactiva descargas pesadas.
- `FORCE_REDOWNLOAD`: fuerza volver a descargar aunque el archivo ya exista.

Por seguridad, todos empiezan en `False`.

Cuando queramos reconstruir los datos desde cero, activaremos estos parámetros de forma consciente.


In [37]:
RUN_DOWNLOADS = False
RUN_HEAVY_DOWNLOADS = False
FORCE_REDOWNLOAD = False

HTTP_TIMEOUT_SECONDS = 60

print("Parámetros de ejecución:")
print(f"RUN_DOWNLOADS: {RUN_DOWNLOADS}")
print(f"RUN_HEAVY_DOWNLOADS: {RUN_HEAVY_DOWNLOADS}")
print(f"FORCE_REDOWNLOAD: {FORCE_REDOWNLOAD}")
print(f"HTTP_TIMEOUT_SECONDS: {HTTP_TIMEOUT_SECONDS}")


Parámetros de ejecución:
RUN_DOWNLOADS: False
RUN_HEAVY_DOWNLOADS: False
FORCE_REDOWNLOAD: False
HTTP_TIMEOUT_SECONDS: 60


### 8.2 Catálogo de fuentes raw esperadas

Esta celda define el catálogo de archivos originales esperados.

Cada registro indica:

- nombre del dataset,
- entidad fuente,
- URL oficial o endpoint de referencia,
- ruta local esperada,
- si requiere API Key,
- si es una descarga pesada,
- si es necesario para construir la base de modelado.

Este catálogo no descarga datos. Solo documenta el inventario de fuentes originales.


In [38]:
@dataclass(frozen=True)
class RawDatasetSpec:
    """Describe una fuente raw esperada en el proyecto."""

    dataset: str
    source_owner: str
    source_url: str
    local_path: Path
    requires_api_key: bool
    is_heavy_download: bool
    required_for_modeling_base: bool
    access_note: str


class RawDatasetCatalog:
    """Catálogo de fuentes raw del proyecto."""

    def __init__(self, specs: list[RawDatasetSpec]):
        self.specs = specs

    def to_dataframe(self) -> pd.DataFrame:
        records = []

        for spec in self.specs:
            exists = spec.local_path.exists()
            size_mb = round(spec.local_path.stat().st_size / 1024 / 1024, 2) if exists else 0

            records.append(
                {
                    "Dataset": spec.dataset,
                    "Fuente": spec.source_owner,
                    "URL o endpoint de referencia": spec.source_url,
                    "Ruta raw esperada": str(spec.local_path.relative_to(config.project_root)),
                    "Existe localmente": exists,
                    "Tamaño MB": size_mb,
                    "Requiere API Key": spec.requires_api_key,
                    "Descarga pesada": spec.is_heavy_download,
                    "Necesario para base de modelado": spec.required_for_modeling_base,
                    "Nota de acceso": spec.access_note,
                }
            )

        return pd.DataFrame(records)


raw_dataset_specs = [
    RawDatasetSpec(
        dataset="GBFS estaciones actuales BiciMAD",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://madrid.publicbikesystem.net/customer/gbfs/v2/gbfs.json",
        local_path=config.raw_bicimad_dir / "gbfs_station_information_raw.json",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Se obtiene a partir del feed GBFS. Útil para referencia actual, mapa o demo.",
    ),
    RawDatasetSpec(
        dataset="GBFS estado actual BiciMAD",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://madrid.publicbikesystem.net/customer/gbfs/v2/gbfs.json",
        local_path=config.raw_bicimad_dir / "gbfs_station_status_snapshot_raw.json",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Se obtiene a partir del feed GBFS. Representa una fotografía actual de disponibilidad.",
    ),
    RawDatasetSpec(
        dataset="Histórico diario de viajes BiciMAD",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://datos.emtmadrid.es/dataset/historico-de-viajes-de-bicimad",
        local_path=config.raw_bicimad_dir / "bicimad_daily_trips_raw.csv",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Serie diaria de viajes. Sirve como contexto de negocio, no como base principal de entrenamiento.",
    ),
    RawDatasetSpec(
        dataset="Histórico de estaciones BiciMAD 2022",
        source_owner="EMT Madrid / BiciMAD",
        source_url="https://datos.emtmadrid.es/es/dataset/historicos-de-bicimad-2017_2023",
        local_path=config.raw_bicimad_dir / "bicimad_station_status_history_2022_raw.zip",
        requires_api_key=False,
        is_heavy_download=True,
        required_for_modeling_base=True,
        access_note="Descarga pesada. Contiene histórico de viajes anonimizados y estado de estaciones. Se usará la parte de estado de estaciones 2022.",
    ),
    RawDatasetSpec(
        dataset="Calendario laboral de Madrid",
        source_owner="Ayuntamiento de Madrid",
        source_url="https://datos.madrid.es/dataset/300082-0-calendario_laboral",
        local_path=config.raw_calendar_dir / "madrid_working_calendar_raw.csv",
        requires_api_key=False,
        is_heavy_download=False,
        required_for_modeling_base=True,
        access_note="Calendario laboral 2013-2026. Se usará para identificar laborables, fines de semana y festivos.",
    ),
    RawDatasetSpec(
        dataset="Inventario de estaciones AEMET",
        source_owner="AEMET OpenData",
        source_url="https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones",
        local_path=config.raw_weather_dir / "aemet_stations_inventory_raw.json",
        requires_api_key=True,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Permite seleccionar estaciones meteorológicas cercanas y fiables para Madrid.",
    ),
    RawDatasetSpec(
        dataset="Climatología diaria AEMET 2022",
        source_owner="AEMET OpenData",
        source_url="https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_inicio}/fechafin/{fecha_fin}/estacion/{indicativo}",
        local_path=config.raw_weather_dir / "aemet_daily_climate_selected_stations_2022_raw.json",
        requires_api_key=True,
        is_heavy_download=False,
        required_for_modeling_base=True,
        access_note="Datos meteorológicos diarios observados. Se descargarán por rangos y estaciones seleccionadas.",
    ),
    RawDatasetSpec(
        dataset="Predicción horaria AEMET Madrid",
        source_owner="AEMET OpenData",
        source_url="https://opendata.aemet.es/opendata/api/prediccion/especifica/municipio/horaria/28079",
        local_path=config.raw_weather_dir / "aemet_forecast_hourly_madrid_raw.json",
        requires_api_key=True,
        is_heavy_download=False,
        required_for_modeling_base=False,
        access_note="Predicción horaria para Madrid. Útil para demo futura, no para entrenar el histórico 2022.",
    ),
]

raw_catalog = RawDatasetCatalog(raw_dataset_specs)
raw_catalog_df = raw_catalog.to_dataframe()

display(raw_catalog_df)


,Dataset,Fuente,URL o endpoint de referencia,Ruta raw esperada,Existe localmente,Tamaño MB,Requiere API Key,Descarga pesada,Necesario para base de modelado,Nota de acceso
0,GBFS estaciones actuales BiciMAD,EMT Madrid / BiciMAD,https://madrid.publicbikesystem.net/customer/g...,data/raw/bicimad/gbfs_station_information_raw....,False,0,False,False,False,Se obtiene a partir del feed GBFS. Útil para r...
1,GBFS estado actual BiciMAD,EMT Madrid / BiciMAD,https://madrid.publicbikesystem.net/customer/g...,data/raw/bicimad/gbfs_station_status_snapshot_...,False,0,False,False,False,Se obtiene a partir del feed GBFS. Representa ...
2,Histórico diario de viajes BiciMAD,EMT Madrid / BiciMAD,https://datos.emtmadrid.es/dataset/historico-d...,data/raw/bicimad/bicimad_daily_trips_raw.csv,False,0,False,False,False,Serie diaria de viajes. Sirve como contexto de...
3,Histórico de estaciones BiciMAD 2022,EMT Madrid / BiciMAD,https://datos.emtmadrid.es/es/dataset/historic...,data/raw/bicimad/bicimad_station_status_histor...,False,0,False,True,True,Descarga pesada. Contiene histórico de viajes ...
4,Calendario laboral de Madrid,Ayuntamiento de Madrid,https://datos.madrid.es/dataset/300082-0-calen...,data/raw/calendar/madrid_working_calendar_raw.csv,False,0,False,False,True,Calendario laboral 2013-2026. Se usará para id...
5,Inventario de estaciones AEMET,AEMET OpenData,https://opendata.aemet.es/opendata/api/valores...,data/raw/weather/aemet_stations_inventory_raw....,False,0,True,False,False,Permite seleccionar estaciones meteorológicas ...
6,Climatología diaria AEMET 2022,AEMET OpenData,https://opendata.aemet.es/opendata/api/valores...,data/raw/weather/aemet_daily_climate_selected_...,False,0,True,False,True,Datos meteorológicos diarios observados. Se de...
7,Predicción horaria AEMET Madrid,AEMET OpenData,https://opendata.aemet.es/opendata/api/predicc...,data/raw/weather/aemet_forecast_hourly_madrid_...,False,0,True,False,False,Predicción horaria para Madrid. Útil para demo...


### 8.3 Verificación local de archivos raw

Esta celda comprueba qué archivos originales existen ya en `data/raw`.

Es normal que falten archivos al clonar el repositorio, porque los datos reales no se suben a GitHub.

La verificación permite saber qué está listo y qué tendrá que descargarse o regenerarse después.


In [39]:
class RawDataAvailabilityChecker:
    """Comprueba la disponibilidad local de archivos raw."""

    def __init__(self, raw_catalog_df: pd.DataFrame):
        self.raw_catalog_df = raw_catalog_df.copy()

    def summarize(self) -> pd.DataFrame:
        summary = (
            self.raw_catalog_df
            .groupby(["Necesario para base de modelado", "Requiere API Key", "Descarga pesada"], dropna=False)
            .agg(
                archivos_esperados=("Dataset", "count"),
                archivos_existentes=("Existe localmente", "sum"),
                tamano_total_mb=("Tamaño MB", "sum"),
            )
            .reset_index()
        )

        summary["archivos_pendientes"] = (
            summary["archivos_esperados"] - summary["archivos_existentes"]
        )

        return summary

    def missing_files(self) -> pd.DataFrame:
        return self.raw_catalog_df[~self.raw_catalog_df["Existe localmente"]].copy()

    def missing_required_files(self) -> pd.DataFrame:
        return self.raw_catalog_df[
            (~self.raw_catalog_df["Existe localmente"])
            & (self.raw_catalog_df["Necesario para base de modelado"])
        ].copy()


raw_availability_checker = RawDataAvailabilityChecker(raw_catalog_df)

raw_availability_summary_df = raw_availability_checker.summarize()
missing_raw_files_df = raw_availability_checker.missing_files()
missing_required_raw_files_df = raw_availability_checker.missing_required_files()

print("Resumen de disponibilidad de archivos raw:")
display(raw_availability_summary_df)

print("\nArchivos raw pendientes:")
display(
    missing_raw_files_df[
        [
            "Dataset",
            "Ruta raw esperada",
            "Requiere API Key",
            "Descarga pesada",
            "Necesario para base de modelado",
        ]
    ]
)


Resumen de disponibilidad de archivos raw:


,Necesario para base de modelado,Requiere API Key,Descarga pesada,archivos_esperados,archivos_existentes,tamano_total_mb,archivos_pendientes
0,False,False,False,3,0,0,3
1,False,True,False,2,0,0,2
2,True,False,False,1,0,0,1
3,True,False,True,1,0,0,1
4,True,True,False,1,0,0,1



Archivos raw pendientes:


,Dataset,Ruta raw esperada,Requiere API Key,Descarga pesada,Necesario para base de modelado
0,GBFS estaciones actuales BiciMAD,data/raw/bicimad/gbfs_station_information_raw....,False,False,False
1,GBFS estado actual BiciMAD,data/raw/bicimad/gbfs_station_status_snapshot_...,False,False,False
2,Histórico diario de viajes BiciMAD,data/raw/bicimad/bicimad_daily_trips_raw.csv,False,False,False
3,Histórico de estaciones BiciMAD 2022,data/raw/bicimad/bicimad_station_status_histor...,False,True,True
4,Calendario laboral de Madrid,data/raw/calendar/madrid_working_calendar_raw.csv,False,False,True
5,Inventario de estaciones AEMET,data/raw/weather/aemet_stations_inventory_raw....,True,False,False
6,Climatología diaria AEMET 2022,data/raw/weather/aemet_daily_climate_selected_...,True,False,True
7,Predicción horaria AEMET Madrid,data/raw/weather/aemet_forecast_hourly_madrid_...,True,False,False


### 8.4 Resumen de disponibilidad antes de descargar

Esta celda resume el estado actual antes de cualquier descarga.

Si faltan archivos necesarios para la base de modelado, no significa que el proyecto esté mal. Significa que el entorno local todavía no ha reconstruido los datos.

Las descargas se activarán de forma explícita en una sección posterior.


In [40]:
total_raw_files = len(raw_catalog_df)
existing_raw_files = int(raw_catalog_df["Existe localmente"].sum())
missing_raw_files = total_raw_files - existing_raw_files

total_required_raw_files = int(raw_catalog_df["Necesario para base de modelado"].sum())
existing_required_raw_files = int(
    raw_catalog_df[
        raw_catalog_df["Necesario para base de modelado"]
    ]["Existe localmente"].sum()
)
missing_required_raw_files = total_required_raw_files - existing_required_raw_files

print("Estado general de archivos raw:")
print(f"Archivos raw esperados: {total_raw_files}")
print(f"Archivos raw existentes: {existing_raw_files}")
print(f"Archivos raw pendientes: {missing_raw_files}")

print("\nEstado de archivos raw necesarios para la base de modelado:")
print(f"Archivos necesarios esperados: {total_required_raw_files}")
print(f"Archivos necesarios existentes: {existing_required_raw_files}")
print(f"Archivos necesarios pendientes: {missing_required_raw_files}")

print("\nParámetros de descarga actuales:")
print(f"RUN_DOWNLOADS: {RUN_DOWNLOADS}")
print(f"RUN_HEAVY_DOWNLOADS: {RUN_HEAVY_DOWNLOADS}")
print(f"FORCE_REDOWNLOAD: {FORCE_REDOWNLOAD}")

if not RUN_DOWNLOADS:
    print("\nLas descargas generales están desactivadas. Esta ejecución solo verifica archivos.")

if not RUN_HEAVY_DOWNLOADS:
    print("Las descargas pesadas están desactivadas. El histórico grande de BiciMAD no se descargará automáticamente.")


Estado general de archivos raw:
Archivos raw esperados: 8
Archivos raw existentes: 0
Archivos raw pendientes: 8

Estado de archivos raw necesarios para la base de modelado:
Archivos necesarios esperados: 3
Archivos necesarios existentes: 0
Archivos necesarios pendientes: 3

Parámetros de descarga actuales:
RUN_DOWNLOADS: False
RUN_HEAVY_DOWNLOADS: False
FORCE_REDOWNLOAD: False

Las descargas generales están desactivadas. Esta ejecución solo verifica archivos.
Las descargas pesadas están desactivadas. El histórico grande de BiciMAD no se descargará automáticamente.


## 9. Validación inicial de datos raw

En esta sección se validan los archivos originales que existan localmente en `data/raw`.

La validación inicial no transforma los datos. Su objetivo es comprobar que los archivos descargados pueden leerse correctamente y que tienen un formato coherente con lo esperado.

Esta fase ayuda a detectar problemas antes de empezar la limpieza:

- archivos que faltan,
- archivos vacíos,
- JSON mal formados,
- CSV que no se pueden leer,
- ZIP corruptos o inesperados,
- archivos necesarios para modelado que todavía no están disponibles.

Si un archivo no existe, no se considera un error del notebook. Simplemente queda marcado como pendiente de descarga.


### 9.1 Objetivo de la validación inicial

La validación inicial responde a una pregunta sencilla:

> ¿Los archivos originales están disponibles y se pueden leer?

Todavía no se revisa la calidad estadística de los datos, ni duplicados, ni nulos, ni rangos de fechas. Eso se hará después, en las secciones de limpieza específicas de cada dataset.

En esta sección solo se comprueba la salud técnica básica de los archivos raw.


### 9.2 Validador general de archivos raw

Esta celda define una clase para validar archivos raw de forma homogénea.

El validador detecta el tipo de archivo según su extensión:

- `.json`: intenta leer el archivo como JSON.
- `.csv`: intenta leer unas primeras filas como CSV.
- `.zip`: revisa que el archivo ZIP pueda abrirse y lista su contenido básico.

Para archivos ZIP grandes no se extrae el contenido ni se realiza una validación profunda por defecto, para evitar tiempos de ejecución innecesarios.


In [41]:
import json
import zipfile


@dataclass
class RawFileValidationResult:
    """Resultado de validación técnica básica de un archivo raw."""

    dataset: str
    path: str
    expected_type: str
    exists: bool
    size_mb: float
    can_be_read: bool
    status: str
    details: str


class RawFileValidator:
    """Valida archivos raw sin modificar su contenido."""

    def __init__(self, spec: RawDatasetSpec, project_root: Path):
        self.spec = spec
        self.project_root = project_root
        self.path = spec.local_path

    def detect_expected_type(self) -> str:
        suffix = self.path.suffix.lower()

        if suffix == ".json":
            return "json"

        if suffix == ".csv":
            return "csv"

        if suffix == ".zip":
            return "zip"

        return "unknown"

    def validate(self) -> RawFileValidationResult:
        expected_type = self.detect_expected_type()

        if not self.path.exists():
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=False,
                size_mb=0,
                can_be_read=False,
                status="pendiente",
                details="El archivo no existe localmente. Debe descargarse o generarse después.",
            )

        size_mb = round(self.path.stat().st_size / 1024 / 1024, 2)

        if size_mb == 0:
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type=expected_type,
                exists=True,
                size_mb=size_mb,
                can_be_read=False,
                status="error",
                details="El archivo existe pero está vacío.",
            )

        if expected_type == "json":
            return self._validate_json(size_mb)

        if expected_type == "csv":
            return self._validate_csv(size_mb)

        if expected_type == "zip":
            return self._validate_zip(size_mb)

        return RawFileValidationResult(
            dataset=self.spec.dataset,
            path=str(self.path.relative_to(self.project_root)),
            expected_type=expected_type,
            exists=True,
            size_mb=size_mb,
            can_be_read=False,
            status="revisar",
            details="Extensión no reconocida para validación automática.",
        )

    def _validate_json(self, size_mb: float) -> RawFileValidationResult:
        try:
            with self.path.open("r", encoding="utf-8") as file:
                data = json.load(file)

            if isinstance(data, dict):
                preview = list(data.keys())[:10]
                details = f"JSON válido tipo dict. Claves principales detectadas: {preview}"
            elif isinstance(data, list):
                details = f"JSON válido tipo list. Elementos detectados: {len(data)}"
            else:
                details = f"JSON válido. Tipo raíz detectado: {type(data).__name__}"

            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type="json",
                exists=True,
                size_mb=size_mb,
                can_be_read=True,
                status="ok",
                details=details,
            )

        except Exception as error:
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type="json",
                exists=True,
                size_mb=size_mb,
                can_be_read=False,
                status="error",
                details=f"No se pudo leer como JSON: {error}",
            )

    def _validate_csv(self, size_mb: float) -> RawFileValidationResult:
        try:
            sample = pd.read_csv(self.path, sep=None, engine="python", nrows=5)

            details = (
                f"CSV legible. Columnas detectadas: {list(sample.columns)}. "
                f"Filas de muestra leídas: {len(sample)}"
            )

            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type="csv",
                exists=True,
                size_mb=size_mb,
                can_be_read=True,
                status="ok",
                details=details,
            )

        except Exception as error:
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type="csv",
                exists=True,
                size_mb=size_mb,
                can_be_read=False,
                status="error",
                details=f"No se pudo leer como CSV: {error}",
            )

    def _validate_zip(self, size_mb: float) -> RawFileValidationResult:
        try:
            with zipfile.ZipFile(self.path, "r") as zip_file:
                file_names = zip_file.namelist()
                preview = file_names[:10]
                total_files = len(file_names)

            details = (
                f"ZIP legible. Archivos internos detectados: {total_files}. "
                f"Primeros elementos: {preview}"
            )

            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type="zip",
                exists=True,
                size_mb=size_mb,
                can_be_read=True,
                status="ok",
                details=details,
            )

        except Exception as error:
            return RawFileValidationResult(
                dataset=self.spec.dataset,
                path=str(self.path.relative_to(self.project_root)),
                expected_type="zip",
                exists=True,
                size_mb=size_mb,
                can_be_read=False,
                status="error",
                details=f"No se pudo abrir como ZIP: {error}",
            )


### 9.3 Validación de archivos raw disponibles

Esta celda aplica el validador a todos los archivos definidos en el catálogo raw.

Si un archivo existe, se intenta leer de forma básica.

Si un archivo no existe, queda marcado como pendiente.

El resultado permite saber qué fuentes están listas para pasar a limpieza y cuáles requieren descarga previa.


In [42]:
if "raw_dataset_specs" not in globals():
    raise RuntimeError(
        "No se ha encontrado raw_dataset_specs. "
        "Ejecuta primero la sección 8 para crear el catálogo de fuentes raw."
    )

raw_validation_results = [
    RawFileValidator(spec=spec, project_root=config.project_root).validate()
    for spec in raw_dataset_specs
]

raw_validation_df = pd.DataFrame(
    [
        {
            "Dataset": result.dataset,
            "Ruta": result.path,
            "Tipo esperado": result.expected_type,
            "Existe": result.exists,
            "Tamaño MB": result.size_mb,
            "Se puede leer": result.can_be_read,
            "Estado": result.status,
            "Detalle": result.details,
        }
        for result in raw_validation_results
    ]
)

display(raw_validation_df)


,Dataset,Ruta,Tipo esperado,Existe,Tamaño MB,Se puede leer,Estado,Detalle
0,GBFS estaciones actuales BiciMAD,data/raw/bicimad/gbfs_station_information_raw....,json,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...
1,GBFS estado actual BiciMAD,data/raw/bicimad/gbfs_station_status_snapshot_...,json,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...
2,Histórico diario de viajes BiciMAD,data/raw/bicimad/bicimad_daily_trips_raw.csv,csv,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...
3,Histórico de estaciones BiciMAD 2022,data/raw/bicimad/bicimad_station_status_histor...,zip,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...
4,Calendario laboral de Madrid,data/raw/calendar/madrid_working_calendar_raw.csv,csv,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...
5,Inventario de estaciones AEMET,data/raw/weather/aemet_stations_inventory_raw....,json,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...
6,Climatología diaria AEMET 2022,data/raw/weather/aemet_daily_climate_selected_...,json,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...
7,Predicción horaria AEMET Madrid,data/raw/weather/aemet_forecast_hourly_madrid_...,json,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...


### 9.4 Resumen de archivos necesarios para modelado

Esta celda filtra la validación para centrarse en los archivos raw necesarios para construir la base enriquecida de modelado.

En este proyecto, los archivos raw necesarios para esa base son:

- histórico de estado de estaciones BiciMAD 2022,
- calendario laboral de Madrid,
- climatología diaria AEMET 2022.

Si alguno falta, se deberá descargar o generar antes de construir la base enriquecida.


In [43]:
required_raw_paths = {
    str(spec.local_path.relative_to(config.project_root))
    for spec in raw_dataset_specs
    if spec.required_for_modeling_base
}

required_raw_validation_df = raw_validation_df[
    raw_validation_df["Ruta"].isin(required_raw_paths)
].copy()

display(required_raw_validation_df)

total_required = len(required_raw_validation_df)
available_required = int(required_raw_validation_df["Existe"].sum())
readable_required = int(required_raw_validation_df["Se puede leer"].sum())
pending_required = total_required - available_required
unreadable_required = available_required - readable_required

print("Resumen de raw necesarios para la base de modelado:")
print(f"Archivos necesarios esperados: {total_required}")
print(f"Archivos necesarios existentes: {available_required}")
print(f"Archivos necesarios legibles: {readable_required}")
print(f"Archivos necesarios pendientes: {pending_required}")
print(f"Archivos necesarios con error de lectura: {unreadable_required}")


,Dataset,Ruta,Tipo esperado,Existe,Tamaño MB,Se puede leer,Estado,Detalle
3,Histórico de estaciones BiciMAD 2022,data/raw/bicimad/bicimad_station_status_histor...,zip,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...
4,Calendario laboral de Madrid,data/raw/calendar/madrid_working_calendar_raw.csv,csv,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...
6,Climatología diaria AEMET 2022,data/raw/weather/aemet_daily_climate_selected_...,json,False,0,False,pendiente,El archivo no existe localmente. Debe descarga...


Resumen de raw necesarios para la base de modelado:
Archivos necesarios esperados: 3
Archivos necesarios existentes: 0
Archivos necesarios legibles: 0
Archivos necesarios pendientes: 3
Archivos necesarios con error de lectura: 0


### 9.5 Interpretación antes de limpiar

Esta sección todavía no limpia datos.

La interpretación debe hacerse así:

- `ok`: el archivo existe y se puede leer técnicamente.
- `pendiente`: el archivo no existe localmente y deberá descargarse o generarse.
- `error`: el archivo existe, pero no se pudo leer correctamente.
- `revisar`: el archivo existe, pero su tipo no está contemplado por el validador automático.

Un archivo marcado como `ok` no significa que sus datos ya estén limpios. Solo significa que el archivo original puede abrirse y está listo para pasar a la fase de limpieza específica.

La limpieza real empieza en el apartado 10 para datasets auxiliares y en el apartado 11 para el histórico principal de BiciMAD 2022.


In [44]:
validation_status_summary = (
    raw_validation_df
    .groupby("Estado", dropna=False)
    .agg(
        archivos=("Dataset", "count"),
        tamano_total_mb=("Tamaño MB", "sum"),
    )
    .reset_index()
    .sort_values("Estado")
)

display(validation_status_summary)

if (raw_validation_df["Estado"] == "error").any():
    print("Hay archivos raw con errores de lectura. Deben revisarse antes de limpiar.")
elif (raw_validation_df["Estado"] == "pendiente").any():
    print("Hay archivos raw pendientes. Es normal si todavía no se han descargado todos los datos.")
else:
    print("Todos los archivos raw definidos existen y se pueden leer técnicamente.")


,Estado,archivos,tamano_total_mb
0,pendiente,8,0


Hay archivos raw pendientes. Es normal si todavía no se han descargado todos los datos.
